# 13 - External Validation

## Objective

Evaluate the trained YAMNet classifier on completely unseen underwater recordings.

These recordings were **never used** during training, validation or testing.

Pipeline

New Audio

↓

YAMNet

↓

Embedding

↓

Classifier

↓

Prediction

↓

Confidence Score

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q tensorflow tensorflow_hub librosa soundfile joblib tqdm

In [3]:
import os
import joblib
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

from pathlib import Path
from tqdm import tqdm

In [4]:
MODEL_PATH = "/content/drive/MyDrive/Underwater Audio Data/models/yamnet_classifier.keras"

ENCODER_PATH = "/content/drive/MyDrive/Underwater Audio Data/embeddings/label_encoder.pkl"

TARGET_SR = 16000

In [7]:
print("Loading model...")

model = tf.keras.models.load_model(MODEL_PATH)

encoder = joblib.load(ENCODER_PATH)

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

print("Everything loaded successfully!")

Loading model...
Everything loaded successfully!


In [8]:
def load_audio(filepath):

    waveform, _ = librosa.load(
        filepath,
        sr=TARGET_SR,
        mono=True
    )

    return waveform.astype(np.float32)

In [9]:
def extract_embedding(filepath):

    waveform = load_audio(filepath)

    scores, embeddings, spectrogram = yamnet(waveform)

    embeddings = embeddings.numpy()

    return np.mean(embeddings, axis=0)

In [10]:
def predict_audio(filepath):

    embedding = extract_embedding(filepath)

    embedding = embedding.reshape(1, -1)

    probabilities = model.predict(
        embedding,
        verbose=0
    )[0]

    prediction = np.argmax(probabilities)

    predicted_label = encoder.inverse_transform([prediction])[0]

    confidence = probabilities[prediction]

    return predicted_label, confidence, probabilities

In [21]:
from google.colab import files

uploaded = files.upload()

AUDIO_FILE = next(iter(uploaded))

print(f"Uploaded file: {AUDIO_FILE}")

Saving flutie8211-big-ship-horn-blast-546920.wav to flutie8211-big-ship-horn-blast-546920.wav
Uploaded file: flutie8211-big-ship-horn-blast-546920.wav


In [22]:
from IPython.display import Audio

Audio(AUDIO_FILE)

In [23]:
label, confidence, probabilities = predict_audio(AUDIO_FILE)

print("=" * 40)
print("Prediction")
print("=" * 40)

print(f"Predicted Class : {label}")
print(f"Confidence      : {confidence:.2%}")

Prediction
Predicted Class : Vessels
Confidence      : 99.86%


In [24]:
print("\nClass Probabilities\n")

for cls, prob in zip(encoder.classes_, probabilities):
    print(f"{cls:<12}: {prob:.2%}")


Class Probabilities

Ambience    : 0.10%
Biological  : 0.05%
Vessels     : 99.86%
